# Orbit Wars Agent Submission

## Goal

Package the current Orbit Wars agent candidate, run a fixed-seed smoke benchmark on Kaggle, and write a submission archive.

This notebook is intentionally version-agnostic. Keep agent history and decisions in `docs/04_agent_version_log.md`; change only `CFG["agent_version"]` when promoting a new candidate.

## Artifacts Written

- `main.py`
- `submission.tar.gz`
- `agent_submission_environment_info.json`
- `agent_submission_seed_summary.csv`
- `agent_submission_run_errors.csv`
- `agent_submission_findings.md`


## 1. Runtime Setup

Select the active agent version, locate its `main.py`, and copy it to the Kaggle working directory. On Kaggle, the uploaded kernel folder should include a root-level `main.py`; locally, the notebook can read from `agents/<agent_version>/main.py`.


In [ ]:
import importlib.util
import json
import platform
import shutil
import sys
import tarfile
import traceback
from pathlib import Path
from typing import Any, Optional, Sequence

import pandas as pd

CFG = {
    "agent_version": "roi_reserve_v2",
    "n_seeds": 30,
    "primary_opponent": "random",
    "fallback_opponent": "passive",
    "run_simulations": True,
    "write_submission": True,
}

IS_KAGGLE = Path("/kaggle").exists()
WORKING = Path("/kaggle/working") if IS_KAGGLE else Path("outputs/agent_submission")
WORKING.mkdir(parents=True, exist_ok=True)

AGENT_VERSION = str(CFG["agent_version"])
SOURCE_CANDIDATES = [
    Path("main.py"),
    Path("..") / "agents" / AGENT_VERSION / "main.py",
    Path("agents") / AGENT_VERSION / "main.py",
    Path("/kaggle/working/main.py"),
]


def first_existing(paths: Sequence[Path]) -> Optional[Path]:
    """Return the first existing path from ordered candidates."""
    for path in paths:
        if path.exists():
            return path
    return None


SOURCE_MAIN = first_existing(SOURCE_CANDIDATES)
if SOURCE_MAIN is None:
    searched = "\n".join(str(path) for path in SOURCE_CANDIDATES)
    raise FileNotFoundError(f"Could not find agent source for {AGENT_VERSION}. Searched:\n{searched}")


SUBMISSION_MAIN = WORKING / "main.py"
if SOURCE_MAIN.resolve() != SUBMISSION_MAIN.resolve():
    shutil.copy2(SOURCE_MAIN, SUBMISSION_MAIN)

print("is_kaggle:", IS_KAGGLE)
print("working:", WORKING)
print("agent_version:", AGENT_VERSION)
print("source_main:", SOURCE_MAIN)
print("submission_main:", SUBMISSION_MAIN)
print("cfg:", json.dumps(CFG, indent=2))


## 2. Static Verification

Compile the copied agent and run a deterministic action-format check before using the Kaggle environment.


In [ ]:
compile_result = None
try:
    compile(SUBMISSION_MAIN.read_text(), str(SUBMISSION_MAIN), "exec")
    compile_result = "ok"
except Exception:
    compile_result = traceback.format_exc()
    raise


def load_agent(path: Path) -> Any:
    """Load an agent module from a Python file."""
    spec = importlib.util.spec_from_file_location("submission_agent", path)
    module = importlib.util.module_from_spec(spec)
    if spec.loader is None:
        raise ImportError(f"Cannot load {path}")
    spec.loader.exec_module(module)
    return module


def run_synthetic_action_check(agent_module: Any) -> list:
    """Run a deterministic action-format smoke check."""
    obs = {
        "player": 0,
        "planets": [
            [0, 0, 10.0, 10.0, 1.0, 80, 5],
            [1, -1, 13.0, 10.0, 1.0, 12, 1],
            [2, -1, 24.0, 10.0, 2.0, 10, 5],
        ],
        "fleets": [],
        "angular_velocity": 0.03,
        "initial_planets": [],
        "comet_planet_ids": [],
    }
    moves = agent_module.agent(obs)
    assert isinstance(moves, list)
    for move in moves:
        assert isinstance(move, list)
        assert len(move) == 3
        assert isinstance(move[0], int)
        assert isinstance(move[1], float)
        assert isinstance(move[2], int)
        assert move[2] > 0
    return moves


agent_module = load_agent(SUBMISSION_MAIN)
synthetic_moves = run_synthetic_action_check(agent_module)
print("compile_result:", compile_result)
print("synthetic_moves:", synthetic_moves)


## 3. Kaggle Environment Check

Import Orbit Wars from `kaggle_environments`. Local runs may stop at static verification; Kaggle runs should pass this gate.


In [ ]:
env_info = {
    "python": sys.version,
    "platform": platform.platform(),
    "is_kaggle": IS_KAGGLE,
    "agent_version": AGENT_VERSION,
    "compile_result": compile_result,
    "synthetic_moves": synthetic_moves,
}
ENV_AVAILABLE = False
ENV_ERROR = None

try:
    import kaggle_environments
    from kaggle_environments import make

    ENV_AVAILABLE = True
    env_info["kaggle_environments_version"] = getattr(kaggle_environments, "__version__", "unknown")
except Exception as exc:
    ENV_ERROR = repr(exc)
    env_info["kaggle_environments_error"] = traceback.format_exc()

try:
    if ENV_AVAILABLE:
        make("orbit_wars", configuration={"seed": 0}, debug=True)
        env_info["orbit_wars_available"] = True
except Exception as exc:
    ENV_AVAILABLE = False
    ENV_ERROR = repr(exc)
    env_info["orbit_wars_available"] = False
    env_info["orbit_wars_error"] = traceback.format_exc()

print(json.dumps(env_info, indent=2, default=str))


## 4. Fixed-Seed Benchmark

Run the active agent against `random` over fixed seeds. If `random` is unavailable, fall back to a passive no-op opponent so the notebook still exercises the environment.


In [ ]:
PASSIVE_PATH = WORKING / "passive_agent.py"
PASSIVE_PATH.write_text(
    'def agent(obs):\n'
    '    """Return no actions for fallback benchmark simulations."""\n'
    '    return []\n'
)



def score_from_obs(obs: Any, player: int) -> float:
    """Calculate a final ship-count proxy for one player."""
    if isinstance(obs, dict):
        planets = obs.get("planets", []) or []
        fleets = obs.get("fleets", []) or []
    else:
        planets = getattr(obs, "planets", []) or []
        fleets = getattr(obs, "fleets", []) or []
    planet_score = sum(float(row[5]) for row in planets if int(row[1]) == player)
    fleet_score = sum(float(row[6]) for row in fleets if int(row[1]) == player)
    return planet_score + fleet_score


def summarize_seed(seed: int, env: Any, opponent_label: str) -> dict[str, Any]:
    """Summarize one completed Orbit Wars episode."""
    final_step = env.steps[-1]
    final_obs = final_step[0].observation
    return {
        "seed": seed,
        "opponent": opponent_label,
        "steps": len(env.steps),
        "player0_reward": final_step[0].reward,
        "player0_status": final_step[0].status,
        "player1_reward": final_step[1].reward if len(final_step) > 1 else None,
        "player1_status": final_step[1].status if len(final_step) > 1 else None,
        "player0_score_proxy": score_from_obs(final_obs, 0),
        "player1_score_proxy": score_from_obs(final_obs, 1),
    }


seed_rows = []
run_errors = []
N_SEEDS = int(CFG["n_seeds"])

if ENV_AVAILABLE and CFG["run_simulations"]:
    for seed in range(N_SEEDS):
        try:
            env = make("orbit_wars", configuration={"seed": seed}, debug=True)
            opponent_label = str(CFG["primary_opponent"])
            try:
                env.run([str(SUBMISSION_MAIN), opponent_label])
            except Exception:
                opponent_label = str(CFG["fallback_opponent"])
                env = make("orbit_wars", configuration={"seed": seed}, debug=True)
                env.run([str(SUBMISSION_MAIN), str(PASSIVE_PATH)])
            row = summarize_seed(seed, env, opponent_label)
            seed_rows.append(row)
            print("seed", seed, "reward", row["player0_reward"], "score_proxy", row["player0_score_proxy"])
        except Exception as exc:
            run_errors.append({"seed": seed, "error": repr(exc)})
            print("seed failed", seed, repr(exc))
else:
    print("Orbit Wars environment is unavailable in this runtime. Static verification only.")

seed_df = pd.DataFrame(seed_rows)
errors_df = pd.DataFrame(run_errors)
seed_df.to_csv(WORKING / "agent_submission_seed_summary.csv", index=False)
errors_df.to_csv(WORKING / "agent_submission_run_errors.csv", index=False)

env_info["seed_count_requested"] = N_SEEDS
env_info["seed_count_completed"] = len(seed_rows)
env_info["run_error_count"] = len(run_errors)
with open(WORKING / "agent_submission_environment_info.json", "w") as f:
    json.dump(env_info, f, indent=2, default=str)

wins = int((seed_df["player0_reward"] == 1).sum()) if not seed_df.empty else 0
losses = int((seed_df["player0_reward"] != 1).sum()) if not seed_df.empty else 0
win_rate = wins / len(seed_df) if len(seed_df) else 0.0
findings = f"""# Agent Submission Findings

- Agent version: `{AGENT_VERSION}`
- Orbit Wars environment available: `{ENV_AVAILABLE}`
- Seeds completed: `{len(seed_rows)}` of `{N_SEEDS}`
- Run errors: `{len(run_errors)}`
- Wins vs random: `{wins}`
- Losses vs random: `{losses}`
- Win rate vs random: `{win_rate:.1%}`
"""
(WORKING / "agent_submission_findings.md").write_text(findings)
print(findings)


## 5. Submission Package

Create a root-level `submission.tar.gz` containing `main.py`. Submit the archive from the notebook output after the smoke benchmark is acceptable and the version log in docs is updated.


In [ ]:
if CFG["write_submission"]:
    submission_tar = WORKING / "submission.tar.gz"
    with tarfile.open(submission_tar, "w:gz") as tar:
        tar.add(SUBMISSION_MAIN, arcname="main.py")
    print("wrote:", submission_tar)
else:
    print("write_submission is disabled")

print("wrote:", SUBMISSION_MAIN)
print("wrote:", WORKING / "agent_submission_environment_info.json")
print("wrote:", WORKING / "agent_submission_seed_summary.csv")
print("wrote:", WORKING / "agent_submission_run_errors.csv")
print("wrote:", WORKING / "agent_submission_findings.md")
